# 🎓 EduPath AI — LLM Training with TRL GRPO
**Team KRIYA | Meta Hackathon 2026**

This notebook trains an LLM-based tutoring agent using Group Relative Policy Optimization (GRPO) from HuggingFace TRL.

The EduPath environment serves as the **verifier**: the LLM generates candidate tutoring actions, and the environment scores them.

**Runtime:** Select GPU → T4 (free tier works!)

In [ ]:
# Install dependencies
!pip install -q trl transformers accelerate peft datasets torch
!pip install -q unsloth  # 2x faster training

# Clone the EduPath AI repo
!git clone https://huggingface.co/spaces/degree-checker-01/meta-new-space /content/edupath
%cd /content/edupath

import sys
sys.path.insert(0, '/content/edupath/backend')

## Step 1: Verify the Environment Works

In [ ]:
from environment.env import EduPathEnv
from environment.models import Action, ActionType
from environment.student import student_manager

# Create environment and student
env = EduPathEnv(seed=42)
student = student_manager.create(name='Colab Demo')
student_manager.update_from_onboarding(student.id, {
    'target_field': 'tech',
    'learning_goal': 'Learn Python programming',
    'weekly_hours': 10,
})
obs = env.reset(student_id=student.id, seed=42)

print(f'Student: {student.id}')
print(f'Available topics: {obs.available_topics[:5]}')
print(f'Environment ready! ✅')

## Step 2: Define the Reward Function (Environment as Verifier)

The LLM generates JSON tutoring actions. The environment scores them.

In [ ]:
import json
import re

def edupath_reward(completions, **kwargs):
    """Score LLM completions using the EduPath environment."""
    rewards = []
    for text in completions:
        try:
            # Extract JSON from LLM output
            match = re.search(r'\{[^}]+\}', text)
            if not match:
                rewards.append(-0.2)
                continue
            action_dict = json.loads(match.group())
            
            # Score using environment
            action_type = ActionType(action_dict.get('type', 'recommend_resource'))
            action = Action(
                type=action_type,
                topic_id=action_dict.get('topic_id'),
            )
            reward = env._calculate_reward(action)
            
            # Bonus for valid JSON format
            r = reward.value + 0.1
            rewards.append(r)
        except Exception:
            rewards.append(-0.2)
    return rewards

# Demo: score some candidate actions
test_actions = [
    '{"type": "recommend_topic", "topic_id": "python_basics"}',
    '{"type": "assign_quiz", "topic_id": "python_basics"}',
    '{"type": "mark_job_ready"}',
    'invalid garbage',
]
test_rewards = edupath_reward(test_actions)
for a, r in zip(test_actions, test_rewards):
    print(f'  {"✓" if r > 0 else "✗"} {a[:50]:50s} → reward={r:+.2f}')
print(f'\nReward function working! ✅')

## Step 3: Build Training Prompts from Environment States

In [ ]:
from datasets import Dataset

def make_prompt(obs_dict):
    return f"""You are an AI tutoring agent. Choose the best next action.

STATE:
- Completed: {obs_dict.get('completed_topics', [])}
- Current: {obs_dict.get('current_topic', 'None')}
- Available: {obs_dict.get('available_topics', [])[:5]}
- Quiz scores: {obs_dict.get('quiz_history_summary', {})}
- Job readiness: {obs_dict.get('job_readiness_score', 0)}

ACTIONS: recommend_topic, assign_quiz, assign_mini_project, assign_capstone, recommend_resource, mark_job_ready

Respond with ONLY JSON: {{"type": "<action>", "topic_id": "<id>"}}"""

# Generate 100 diverse prompts
prompts = []
for seed in range(100):
    env_i = EduPathEnv(seed=seed)
    s = student_manager.create(name=f'train_{seed}')
    student_manager.update_from_onboarding(s.id, {
        'target_field': 'tech', 'learning_goal': 'ML Engineer', 'weekly_hours': 10,
    })
    obs_i = env_i.reset(student_id=s.id, seed=seed)
    prompts.append(make_prompt(obs_i.model_dump()))

dataset = Dataset.from_dict({'prompt': prompts})
print(f'Training dataset: {len(dataset)} prompts ✅')
print(f'\nSample prompt:\n{prompts[0][:300]}...')

## Step 4: Train with GRPO

GRPO generates multiple candidate completions per prompt, scores them with the environment reward function, and updates the policy to favor higher-reward actions.

In [ ]:
from trl import GRPOTrainer, GRPOConfig

# Try Unsloth for 2x speed, fall back to standard HF
try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        'unsloth/Qwen2.5-0.5B-Instruct',
        max_seq_length=1024,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=16,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        lora_alpha=16, lora_dropout=0, bias='none',
    )
    print('Using Unsloth (2x faster) ✅')
except ImportError:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(
        'Qwen/Qwen2.5-0.5B-Instruct', torch_dtype='auto', device_map='auto'
    )
    tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print('Using standard HuggingFace ✅')

# GRPO config
config = GRPOConfig(
    output_dir='edupath_grpo',
    max_steps=200,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    logging_steps=10,
    max_completion_length=64,
    num_generations=4,
    report_to='none',
)

trainer = GRPOTrainer(
    model=model,
    args=config,
    train_dataset=dataset,
    processing_class=tokenizer,
    reward_funcs=edupath_reward,
)

print('\nStarting GRPO training...')
trainer.train()
print('\nTraining complete! ✅')

## Step 5: Visualize Reward Improvement

In [ ]:
import matplotlib.pyplot as plt

# Extract training rewards from log
log = trainer.state.log_history
steps = [e['step'] for e in log if 'loss' in e]
losses = [e['loss'] for e in log if 'loss' in e]
rewards_log = [e.get('reward', 0) for e in log if 'reward' in e]
reward_steps = [e['step'] for e in log if 'reward' in e]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(steps, losses, 'b-', linewidth=1.5)
ax1.set_xlabel('Training Step')
ax1.set_ylabel('Loss')
ax1.set_title('GRPO Training Loss')
ax1.grid(True, alpha=0.3)

if reward_steps:
    ax2.plot(reward_steps, rewards_log, 'g-', linewidth=1.5)
    ax2.set_xlabel('Training Step')
    ax2.set_ylabel('Mean Reward')
    ax2.set_title('EduPath Environment Reward')
    ax2.grid(True, alpha=0.3)

plt.suptitle('EduPath AI — GRPO Training on EduPath Environment', fontsize=14)
plt.tight_layout()
plt.savefig('grpo_training_curve.png', dpi=150)
plt.show()
print('Saved: grpo_training_curve.png ✅')

## Step 6: Test the Trained Model

In [ ]:
# Generate actions from the trained model
test_prompt = prompts[0]
inputs = tokenizer(test_prompt, return_tensors='pt').to(model.device)
outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.7, do_sample=True, num_return_sequences=5)

print('Trained model outputs:')
for i, out in enumerate(outputs):
    text = tokenizer.decode(out[inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    reward = edupath_reward([text])[0]
    print(f'  {i+1}. {text.strip()[:80]:80s} → reward={reward:+.2f}')

# Save model
trainer.save_model('edupath_grpo_final')
tokenizer.save_pretrained('edupath_grpo_final')
print('\nModel saved to edupath_grpo_final/ ✅')

## Summary

This notebook demonstrates:
1. **Environment as Verifier**: EduPath AI scores LLM actions via BKT-based student simulation
2. **GRPO Training**: TRL's GRPO generates multiple candidates, environments scores them, policy updates
3. **Reward Improvement**: Training curves show the LLM learning better tutoring strategies
4. **Unsloth Acceleration**: 2x faster training with Unsloth's optimized kernels

The core loop: `prompt → LLM generates action → environment verifies → reward → policy update`